In [ ]:
# cell 1 - install lama parse
!pip install -q llama-parse nest-asyncio

In [ ]:
# Cell 2 — Imports & API key
import os
import asyncio
import nest_asyncio
from pathlib import Path

nest_asyncio.apply()  # needed to run async LlamaParse calls inside a notebook

from llama_parse import LlamaParse

# Get a free key at https://cloud.llamaindex.ai -> API Keys
os.environ["LLAMA_CLOUD_API_KEY"] = ""

In [ ]:
# Cell 3 — Folders
input_dir = Path("docs")
output_dir = Path("clean_data")
output_dir.mkdir(parents=True, exist_ok=True)

pdf_files = list(input_dir.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF(s):")
for f in pdf_files:
    print(f"  - {f.name}")

In [ ]:
# Cell 4 — Configure the parser
parser = LlamaParse(
    result_type="markdown",
    language="ar",
    verbose=True,
    num_workers=4,
    premium_mode=True,   # forces page-image rendering + vision-based extraction
                         # instead of trusting the PDF's (broken) embedded text layer
)

In [ ]:
# Test cell - parse page 1 of doc 0
test_pdf = pdf_files[0]
print("Parsing (test - first 2 pages only):", test_pdf.name)

parser_test = LlamaParse(
    result_type="markdown",
    verbose=True,
    premium_mode=True,
    target_pages="1",   # 0-indexed! this parses only page 1 and 2
)

documents = parser_test.load_data(str(test_pdf))
full_markdown = "\n\n".join(doc.text for doc in documents)

print("--- Preview ---")
print(full_markdown[:1500])

In [ ]:
# Test Cell 6 — Save one file to confirm it looks right
test_output = output_dir / f"{test_pdf.stem}.md"
test_output.write_text(full_markdown, encoding="utf-8")
print(f"Saved to: {test_output}")


In [ ]:
# Cell 7 — Batch process all files
results = []

for pdf_file in pdf_files:
    md_path = output_dir / f"{pdf_file.stem}.md"

    if md_path.exists():
        print(f"⏭️  Skipping {pdf_file.name} (already exists)")
        results.append({"file": pdf_file.name, "status": "skipped", "path": md_path})
        continue

    print(f"Processing: {pdf_file.name}")
    try:
        documents = parser.load_data(str(pdf_file))
        full_markdown = "\n\n".join(doc.text for doc in documents)
        print(f"  📄 {len(documents)} page(s) parsed")


        if not full_markdown.strip():
            print(f"  ❌ Empty result for {pdf_file.name}")
            results.append({"file": pdf_file.name, "status": "empty", "path": None})
            continue

        md_path.write_text(full_markdown, encoding="utf-8")
        print(f"  ✅ Saved: {md_path.name}")
        results.append({"file": pdf_file.name, "status": "success", "path": md_path})

    except Exception as e:
        print(f"  ❌ Error on {pdf_file.name}: {e}")
        results.append({"file": pdf_file.name, "status": "error", "path": None})

print("\n" + "="*40)
success = [r for r in results if r["status"] == "success"]
failed = [r for r in results if r["status"] in ("error", "empty")]
print(f"✅ Success: {len(success)}")
print(f"❌ Failed: {len(failed)}")
if failed:
    print("Failed files:", [r["file"] for r in failed])